##Reading from Bronze

In [0]:

import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window


In [0]:
df = spark.table("workspace.bronze.crm_cust_info")


##Transformation

In [0]:
display(df)

In [0]:
## Trimming string columns

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
display(df)

In [0]:
## Normalize names

df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
        .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
        .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
        .when(F.upper(F.col("cst_gndr")) == "M", "Male")
        .otherwise("n/a")
    )
)

display(df)

In [0]:
##Renaming Columns

NAME_MAPS = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "customer_firstname",
    "cst_lastname": "customer_lastname",
    "cst_marital_status": "customer_marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "create_date"
}

df = df.select([F.col(column).alias(NAME_MAPS.get(column, column)) for column in df.columns])

display(df)

In [0]:
window_spec = Window.partitionBy("customer_key").orderBy(
    F.col("customer_firstname").desc(), # Valid names comes first
    F.col("customer_lastname").desc()
)

df = df.withColumn("rank", F.row_number().over(window_spec)) \
             .filter(F.col("rank") == 1) \
             .drop("rank")
             
display(df.groupBy("customer_key").count().filter("count > 1"))

##Write to Silver 

In [0]:
(
  df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.crm_customers")
)